# Titanic - Machine Learning from Disaster


## IMPORT THƯ VIỆN


In [ ]:
# imports
from pathlib import Path
import json
import sys

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

import wandb
RANDOM_STATE = 42
DATA_DIR = Path.cwd() if (Path.cwd() / "train.csv").exists() else Path.cwd() / "titanic"
print(DATA_DIR.resolve())


C:\Users\kelac\Downloads\HOCTAP\deeplearning\quytrinhhocsau\thuchanh02-titanic\titanic


## 2. LOAD DỮ LIỆU

In [9]:
# load data
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

print(train.shape)
print(test.shape)
print(train.isna().sum()[train.isna().sum() > 0].sort_values(ascending=False))
train.head()


(891, 12)
(418, 11)
Cabin       687
Age         177
Embarked      2
dtype: int64


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 3. Feature engineering

In [10]:
def add_features(data):
    result = data.copy()
    result["Title"] = (
        result["Name"].str.extract(r",\s*([^.]*)\.", expand=False)
        .fillna("Unknown").str.strip()
        .replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    )
    result.loc[~result["Title"].isin({"Mr", "Miss", "Mrs", "Master"}), "Title"] = "Rare"
    result["FamilySize"] = result["SibSp"] + result["Parch"] + 1
    result["IsAlone"] = (result["FamilySize"] == 1).astype(int)
    result["IsChild"] = (result["Age"] < 16).astype(int)
    result["IsMother"] = ((result["Sex"] == "female") & (result["Parch"] > 0) &
                          (result["Age"] > 18) & (result["Title"] == "Mrs")).astype(int)
    result["HasCabin"] = result["Cabin"].notna().astype(int)
    result["Deck"] = result["Cabin"].str[0].fillna("U")
    result["Fare_Pclass"] = result["Fare"] / result["Pclass"]
    result["Age_Pclass"] = result["Age"] * result["Pclass"]
    result["TicketPrefix"] = (result["Ticket"].astype(str)
                               .str.replace(r"\d", "", regex=True)
                               .str.replace(r"[\s./]+", "", regex=True)
                               .replace("", "NONE"))
    return result

feature_columns = ["Pclass", "Sex", "Age", "Fare", "Embarked", "Title",
                   "FamilySize", "IsAlone", "IsChild", "IsMother", "Deck",
                   "HasCabin", "Fare_Pclass", "Age_Pclass", "TicketPrefix",
                   "SibSp", "Parch"]

train_features = add_features(train)
test_features = add_features(test)
X = train_features[feature_columns]
y = train["Survived"]
X_test = test_features[feature_columns]
X.head()


,Pclass,Sex,Age,Fare,Embarked,Title,FamilySize,IsAlone,IsChild,IsMother,Deck,HasCabin,Fare_Pclass,Age_Pclass,TicketPrefix,SibSp,Parch
0,3,male,22.0,7.2500,S,Mr,2,0,0,0,U,0,2.416667,66.0,A,1,0
1,1,female,38.0,71.2833,C,Mrs,2,0,0,0,C,1,71.283300,38.0,PC,1,0
2,3,female,26.0,7.9250,S,Miss,1,1,0,0,U,0,2.641667,78.0,STONO,0,0
3,1,female,35.0,53.1000,S,Mrs,2,0,0,0,C,1,53.100000,35.0,NONE,1,0
4,3,male,35.0,8.0500,S,Mr,1,1,0,0,U,0,2.683333,105.0,NONE,0,0


## 4. FILLING MISSING VALUE

In [11]:
numeric_features = ["Age", "Fare", "FamilySize", "Fare_Pclass", "Age_Pclass", "SibSp", "Parch"]
categorical_features = ["Pclass", "Sex", "Embarked", "Title", "IsAlone", "IsChild",
                       "IsMother", "Deck", "HasCabin", "TicketPrefix"]

def build_preprocessor():
    numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")),
                                 ("scaler", StandardScaler())])
    categorical_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                     ("onehot", OneHotEncoder(handle_unknown="ignore"))])
    return ColumnTransformer([("numeric", numeric_pipeline, numeric_features),
                              ("categorical", categorical_pipeline, categorical_features)])

base_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=400, max_depth=6,
        min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1),
    "SVM": SVC(C=1.0, kernel="rbf", probability=True, random_state=RANDOM_STATE),
}
models = {name: Pipeline([("preprocessor", build_preprocessor()), ("model", model)])
          for name, model in base_models.items()}


## 5. MODEL TRAINING

In [12]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"accuracy": "accuracy", "f1": "f1", "roc_auc": "roc_auc"}
results = []
fitted_models = {}

for name, model in models.items():
    scores = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    model.fit(X_train, y_train)
    probabilities = model.predict_proba(X_valid)[:, 1]
    predictions = (probabilities >= 0.5).astype(int)
    results.append({
        "Model": name,
        "CV_Accuracy": scores["test_accuracy"].mean(),
        "CV_F1": scores["test_f1"].mean(),
        "CV_ROC_AUC": scores["test_roc_auc"].mean(),
        "Validation_Accuracy": accuracy_score(y_valid, predictions),
        "Validation_F1": f1_score(y_valid, predictions),
        "Validation_ROC_AUC": roc_auc_score(y_valid, probabilities),
    })
    fitted_models[name] = model

results_df = pd.DataFrame(results).sort_values(["CV_ROC_AUC", "CV_Accuracy"], ascending=False)
results_df


,Model,CV_Accuracy,CV_F1,CV_ROC_AUC,Validation_Accuracy,Validation_F1,Validation_ROC_AUC
1,Random Forest,0.828278,0.763362,0.877171,0.798883,0.723077,0.846179
0,Logistic Regression,0.829396,0.769350,0.870566,0.815642,0.755556,0.867852
2,SVM,0.831643,0.767882,0.860999,0.832402,0.769231,0.853755


In [13]:
best_name = results_df.iloc[0]["Model"]
best_model = fitted_models[best_name]
print({best_name})
print(classification_report(y_valid, best_model.predict(X_valid)))


{'Random Forest'}
              precision    recall  f1-score   support

           0       0.81      0.87      0.84       110
           1       0.77      0.68      0.72        69

    accuracy                           0.80       179
   macro avg       0.79      0.78      0.78       179
weighted avg       0.80      0.80      0.80       179



## 6. Submission

In [14]:
best_model.fit(X, y)
predictions = (best_model.predict_proba(X_test)[:, 1] >= 0.5).astype(int)
submission = pd.DataFrame({"PassengerId": test["PassengerId"], "Survived": predictions})
submission.to_csv(DATA_DIR / "submission.csv", index=False)
results_df.to_csv(DATA_DIR / "model_comparison.csv", index=False)
joblib.dump(best_model, DATA_DIR / "titanic_model.joblib")
with (DATA_DIR / "model_metadata.json").open("w", encoding="utf-8") as file:
    json.dump({"selected_model": best_name, "random_state": RANDOM_STATE,
               "features": feature_columns, "train_rows": len(train),
               "test_rows": len(test)}, file, ensure_ascii=False, indent=2)

print(DATA_DIR / "submission.csv")
print(len(submission))
submission.head()


c:\Users\kelac\Downloads\HOCTAP\deeplearning\quytrinhhocsau\thuchanh02-titanic\titanic\submission.csv
418


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
